# SAM Decoder vs TinyViT Output Analysis

This notebook explains why `l40s_sam_decoder_summary_pm1_full_b16_e24_s64` underperformed `l40s_tiny_vit_summary_soft_full_e30_s52`, and whether SAM still has useful diversity for an ensemble.

It uses three artifact types when available:

- run metrics: `results/subtask1/vision_runs/<run_id>/metrics.json`
- validation probabilities: `results/subtask1/val_preds/<run_id>_val_probs.npz`
- test submissions: `results/subtask1/submissions/<run_id>.zip`

The key questions are:

1. Did SAM lose because of class collapse, worse boundaries, lower confidence calibration, or the `pm1_ce` target smoothing?
2. Are SAM's correct pixels different enough from TinyViT's correct pixels to justify an ensemble?
3. Does expected-value decoding or probability averaging improve validation Acc±1 enough to deserve a submission candidate?


## Working Hypothesis

The SAM decoder did worse for four likely reasons:

- This is not pretrained RGB SAM. It is a promptless ViT-style encoder plus a custom decoder trained from scratch on Sentinel summary tensors, so it does not inherit SAM's segmentation prior.
- The `pm1_ce` loss deliberately gives credit to adjacent classes. That matches Acc±1, but it can flatten class boundaries and make the model comfortable predicting neighboring labels instead of learning the exact ordinal structure.
- The validation metrics show a severe class-2 failure for SAM. TinyViT is also weak on class 2, but SAM's best checkpoint has class-2 recall at `0.0`, which is a direct loss of useful central-class signal.
- SAM's best epoch was early, suggesting the current architecture/training recipe was not stable enough. TinyViT had stronger validation Acc±1 and then confirmed that strength on CodaBench with `50.63`.

This notebook tests those claims directly from outputs instead of relying on architecture intuition.

In [ ]:
from pathlib import Path
import json
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from PIL import Image
except Exception as exc:
    Image = None
    print('Pillow unavailable; ZIP mask analysis will be skipped:', exc)

ROOT = Path('..') if Path('../results').exists() else Path('.')
RESULTS = ROOT / 'results' / 'subtask1'

TINY_RUN = 'l40s_tiny_vit_summary_soft_full_e30_s52'
SAM_RUN = 'l40s_sam_decoder_summary_pm1_full_b16_e24_s64'

RUNS = {
    'tiny_vit': {
        'run_id': TINY_RUN,
        'public_score': 50.63,
        'notes': 'Full-data TinyViT, summary features, soft ordinal CE, submitted floor.',
    },
    'sam_decoder': {
        'run_id': SAM_RUN,
        'public_score': np.nan,
        'notes': 'Promptless SAM-style encoder with custom dense decoder and pm1_ce.',
    },
}

CLASS_NAMES = ['0', '1', '2', '3', '4']
COLORS = ['#2f4858', '#4f8f8b', '#f2cc5c', '#e07a5f', '#7b2cbf']
plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['axes.grid'] = True


In [ ]:
def load_json(path):
    path = Path(path)
    if not path.exists():
        print('missing:', path)
        return None
    return json.loads(path.read_text())

def metric_path(run_id):
    return RESULTS / 'vision_runs' / run_id / 'metrics.json'

def config_path(run_id):
    return RESULTS / 'vision_runs' / run_id / 'config.json'

def probs_path(run_id):
    return RESULTS / 'val_preds' / f'{run_id}_val_probs.npz'

def zip_path(run_id):
    return RESULTS / 'submissions' / f'{run_id}.zip'

metrics = {name: load_json(metric_path(info['run_id'])) for name, info in RUNS.items()}
configs = {name: load_json(config_path(info['run_id'])) for name, info in RUNS.items()}

rows = []
for name, info in RUNS.items():
    m = metrics.get(name) or {}
    c = configs.get(name) or {}
    rows.append({
        'model': name,
        'run_id': info['run_id'],
        'public_score': info['public_score'],
        'best_epoch': m.get('best_epoch'),
        'val_pm1': m.get('accuracy_pm1'),
        'val_exact': m.get('exact_accuracy'),
        'val_mae': m.get('mean_absolute_error'),
        'loss': c.get('loss'),
        'batch_size': c.get('batch_size'),
        'median_size': c.get('median_size'),
        'input_channels': c.get('input_channels'),
        'notes': info['notes'],
    })

summary = pd.DataFrame(rows)
summary


## Validation Confusion Matrices

The first hard check is per-class recall. A model can have acceptable Acc±1 while still being unusable for an ensemble if it collapses a class or predicts only broad ordinal bands.

In [ ]:
def confusion_df(model_name):
    m = metrics[model_name]
    cm = np.array(m['confusion_matrix'], dtype=np.int64)
    return pd.DataFrame(cm, index=[f'true_{c}' for c in CLASS_NAMES], columns=[f'pred_{c}' for c in CLASS_NAMES])

recall_rows = []
for model_name, m in metrics.items():
    if not m:
        continue
    row = {'model': model_name}
    row.update({f'recall_{k}': v for k, v in m['per_class_recall'].items()})
    recall_rows.append(row)

recalls = pd.DataFrame(recall_rows).set_index('model')
display(recalls)

ax = recalls.T.plot(kind='bar', color=['#2f4858', '#e07a5f'])
ax.set_title('Per-class recall on validation')
ax.set_ylabel('recall')
ax.set_xlabel('class')
plt.ylim(0, 1)
plt.show()

for model_name in metrics:
    if metrics[model_name]:
        print('\n', model_name)
        display(confusion_df(model_name))


In [ ]:
def plot_confusion(model_name, normalize=True):
    cm = np.array(metrics[model_name]['confusion_matrix'], dtype=np.float64)
    if normalize:
        cm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap='viridis')
    ax.set_title(f'{model_name} validation confusion' + (' row-normalized' if normalize else ''))
    ax.set_xticks(range(5), labels=CLASS_NAMES)
    ax.set_yticks(range(5), labels=CLASS_NAMES)
    ax.set_xlabel('predicted')
    ax.set_ylabel('true')
    for i in range(5):
        for j in range(5):
            ax.text(j, i, f'{cm[i, j]:.2f}' if normalize else f'{int(cm[i, j])}', ha='center', va='center', color='white' if cm[i, j] > cm.max() * 0.55 else 'black')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.show()

for name in ['tiny_vit', 'sam_decoder']:
    if metrics.get(name):
        plot_confusion(name)


## Load Validation Probabilities

The `.npz` files contain `patch_ids`, `probs`, `y_true`, and `y_pred`. If SAM artifacts are missing locally, pull them with:

```bash
scripts/runpod_sync.sh --env-file .env.l40s.claude pull \
  /workspace/ai4agri/results/subtask1/val_preds/l40s_sam_decoder_summary_pm1_full_b16_e24_s64_val_probs.npz \
  ./results/subtask1/val_preds/
```

In [ ]:
def load_val_npz(run_id):
    path = probs_path(run_id)
    if not path.exists():
        print('missing validation probabilities:', path)
        return None
    z = np.load(path, allow_pickle=True)
    out = {k: z[k] for k in z.files}
    print(run_id, {k: (v.shape, v.dtype) for k, v in out.items()})
    return out

val = {name: load_val_npz(info['run_id']) for name, info in RUNS.items()}

if val['tiny_vit'] is not None and val['sam_decoder'] is not None:
    assert list(val['tiny_vit']['patch_ids']) == list(val['sam_decoder']['patch_ids'])
    assert np.array_equal(val['tiny_vit']['y_true'], val['sam_decoder']['y_true'])
    print('Validation arrays are aligned.')


In [ ]:
def compute_metrics(y_true, y_pred):
    y_true = y_true.astype(np.int16)
    y_pred = y_pred.astype(np.int16)
    exact = (y_true == y_pred).mean()
    pm1 = (np.abs(y_true - y_pred) <= 1).mean()
    mae = np.abs(y_true - y_pred).mean()
    cm = np.zeros((5, 5), dtype=np.int64)
    for t in range(5):
        mask = y_true == t
        for p in range(5):
            cm[t, p] = np.count_nonzero(mask & (y_pred == p))
    recall = np.diag(cm) / cm.sum(axis=1).clip(min=1)
    return {'exact': exact, 'pm1': pm1, 'mae': mae, 'cm': cm, 'recall': recall}

def expected_decode(probs):
    classes = np.arange(5, dtype=np.float32).reshape(1, 5, 1, 1)
    return np.rint((probs.astype(np.float32) * classes).sum(axis=1)).clip(0, 4).astype(np.uint8)

if val['tiny_vit'] is not None and val['sam_decoder'] is not None:
    y = val['tiny_vit']['y_true']
    decoded_rows = []
    for name in ['tiny_vit', 'sam_decoder']:
        argmax_pred = val[name]['y_pred']
        exp_pred = expected_decode(val[name]['probs'])
        for decode_name, pred in [('argmax_saved', argmax_pred), ('expected_value', exp_pred)]:
            met = compute_metrics(y, pred)
            decoded_rows.append({'model': name, 'decode': decode_name, 'pm1': met['pm1'], 'exact': met['exact'], 'mae': met['mae']})
    display(pd.DataFrame(decoded_rows).sort_values(['pm1', 'exact'], ascending=False))


## Pixel-Level Complementarity

This table is the first ensemble decision point. If `sam_only_pm1` is tiny, SAM adds little beyond TinyViT. If it is non-trivial and concentrated in classes TinyViT misses, SAM may still be worth a low ensemble weight.

In [ ]:
if val['tiny_vit'] is not None and val['sam_decoder'] is not None:
    y = val['tiny_vit']['y_true'].astype(np.int16)
    t_pred = val['tiny_vit']['y_pred'].astype(np.int16)
    s_pred = val['sam_decoder']['y_pred'].astype(np.int16)
    t_ok = np.abs(t_pred - y) <= 1
    s_ok = np.abs(s_pred - y) <= 1
    comp = pd.Series({
        'both_pm1': np.mean(t_ok & s_ok),
        'tiny_only_pm1': np.mean(t_ok & ~s_ok),
        'sam_only_pm1': np.mean(~t_ok & s_ok),
        'both_wrong_pm1': np.mean(~t_ok & ~s_ok),
        'prediction_agreement': np.mean(t_pred == s_pred),
        'mean_abs_model_delta': np.mean(np.abs(t_pred - s_pred)),
    })
    display(comp.to_frame('fraction'))

    per_class = []
    for c in range(5):
        mask = y == c
        per_class.append({
            'class': c,
            'pixels': int(mask.sum()),
            'tiny_pm1': float(t_ok[mask].mean()),
            'sam_pm1': float(s_ok[mask].mean()),
            'sam_minus_tiny_pm1': float(s_ok[mask].mean() - t_ok[mask].mean()),
            'sam_only_pm1': float((~t_ok[mask] & s_ok[mask]).mean()),
            'tiny_only_pm1': float((t_ok[mask] & ~s_ok[mask]).mean()),
        })
    per_class_df = pd.DataFrame(per_class)
    display(per_class_df)
    per_class_df.plot(x='class', y=['tiny_pm1', 'sam_pm1', 'sam_only_pm1'], kind='bar', color=['#2f4858', '#e07a5f', '#7b2cbf'])
    plt.title('Per-class PM1 correctness and SAM-only contribution')
    plt.ylabel('fraction')
    plt.show()


## Probability Calibration And Confidence

`pm1_ce` should spread probability mass to adjacent classes. That can help expected-value decoding, but too much smoothing can lower exact class discrimination and damage rare/middle classes.

In [ ]:
def confidence_frame(model_name):
    z = val[model_name]
    probs = z['probs'].astype(np.float32)
    y = z['y_true']
    pred = z['y_pred']
    top1 = probs.max(axis=1)
    sorted_probs = np.sort(probs, axis=1)
    margin = sorted_probs[:, -1] - sorted_probs[:, -2]
    entropy = -(probs * np.log(probs.clip(1e-7, 1))).sum(axis=1)
    pm1 = np.abs(pred.astype(np.int16) - y.astype(np.int16)) <= 1
    return pd.DataFrame({
        'model': model_name,
        'top1_mean': [float(top1.mean())],
        'margin_mean': [float(margin.mean())],
        'entropy_mean': [float(entropy.mean())],
        'top1_when_pm1_ok': [float(top1[pm1].mean())],
        'top1_when_pm1_wrong': [float(top1[~pm1].mean())],
    })

if val['tiny_vit'] is not None and val['sam_decoder'] is not None:
    conf = pd.concat([confidence_frame('tiny_vit'), confidence_frame('sam_decoder')], ignore_index=True)
    display(conf)

    for name in ['tiny_vit', 'sam_decoder']:
        probs = val[name]['probs'].astype(np.float32)
        top1 = probs.max(axis=1).ravel()
        plt.hist(top1, bins=50, alpha=0.55, label=name, density=True)
    plt.title('Top-1 confidence distribution')
    plt.xlabel('max probability')
    plt.ylabel('density')
    plt.legend()
    plt.show()


## Validation Ensemble Sweep

This tests whether SAM should be used in a final ensemble. `alpha` is the TinyViT weight. If the best row is at or near `alpha=1.0`, SAM is not helping as a probability ensemble. If a smaller alpha wins, inspect the per-class table before spending a submission.

In [ ]:
if val['tiny_vit'] is not None and val['sam_decoder'] is not None:
    y = val['tiny_vit']['y_true']
    tiny_probs = val['tiny_vit']['probs'].astype(np.float32)
    sam_probs = val['sam_decoder']['probs'].astype(np.float32)
    rows = []
    for alpha in np.linspace(0, 1, 21):
        probs = alpha * tiny_probs + (1 - alpha) * sam_probs
        for decode_name, pred in [('argmax', probs.argmax(axis=1).astype(np.uint8)), ('expected', expected_decode(probs))]:
            met = compute_metrics(y, pred)
            rows.append({'tiny_weight': alpha, 'decode': decode_name, 'pm1': met['pm1'], 'exact': met['exact'], 'mae': met['mae']})
    sweep = pd.DataFrame(rows).sort_values(['pm1', 'exact'], ascending=False)
    display(sweep.head(12))

    for decode_name, group in sweep.sort_values('tiny_weight').groupby('decode'):
        plt.plot(group['tiny_weight'], group['pm1'], marker='o', label=decode_name)
    plt.axvline(1.0, color='black', linestyle='--', linewidth=1)
    plt.title('TinyViT/SAM validation probability ensemble')
    plt.xlabel('TinyViT probability weight')
    plt.ylabel('validation Acc±1')
    plt.legend()
    plt.show()


## Per-Patch Error Ranking

Use these patch IDs to inspect visual panels. The first table shows where TinyViT beats SAM most strongly; the second shows where SAM adds unique value.

In [ ]:
if val['tiny_vit'] is not None and val['sam_decoder'] is not None:
    patch_ids = val['tiny_vit']['patch_ids']
    y = val['tiny_vit']['y_true'].astype(np.int16)
    t = val['tiny_vit']['y_pred'].astype(np.int16)
    s = val['sam_decoder']['y_pred'].astype(np.int16)
    t_pm1 = (np.abs(t - y) <= 1).mean(axis=(1, 2))
    s_pm1 = (np.abs(s - y) <= 1).mean(axis=(1, 2))
    patch_df = pd.DataFrame({
        'patch_id': patch_ids,
        'tiny_pm1': t_pm1,
        'sam_pm1': s_pm1,
        'sam_minus_tiny': s_pm1 - t_pm1,
        'model_disagreement': (t != s).mean(axis=(1, 2)),
        'true_majority': [int(np.bincount(arr.ravel(), minlength=5).argmax()) for arr in y],
        'tiny_majority': [int(np.bincount(arr.ravel(), minlength=5).argmax()) for arr in t],
        'sam_majority': [int(np.bincount(arr.ravel(), minlength=5).argmax()) for arr in s],
    })
    display(patch_df.sort_values('sam_minus_tiny').head(15))
    display(patch_df.sort_values('sam_minus_tiny', ascending=False).head(15))


In [ ]:
def show_val_patch(patch_id):
    if val['tiny_vit'] is None or val['sam_decoder'] is None:
        return
    idx = list(val['tiny_vit']['patch_ids']).index(str(patch_id))
    mats = [
        ('true', val['tiny_vit']['y_true'][idx]),
        ('tiny_vit', val['tiny_vit']['y_pred'][idx]),
        ('sam_decoder', val['sam_decoder']['y_pred'][idx]),
        ('abs diff', np.abs(val['tiny_vit']['y_pred'][idx].astype(np.int16) - val['sam_decoder']['y_pred'][idx].astype(np.int16))),
    ]
    fig, axes = plt.subplots(1, 4, figsize=(14, 4))
    for ax, (title, arr) in zip(axes, mats):
        im = ax.imshow(arr, vmin=0, vmax=4, cmap='viridis')
        ax.set_title(title)
        ax.axis('off')
    fig.suptitle(f'Validation patch {patch_id}')
    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.75)
    plt.show()

if val['tiny_vit'] is not None and val['sam_decoder'] is not None:
    for pid in patch_df.sort_values('sam_minus_tiny').head(3)['patch_id']:
        show_val_patch(pid)
    for pid in patch_df.sort_values('sam_minus_tiny', ascending=False).head(3)['patch_id']:
        show_val_patch(pid)


## Test ZIP Distribution And Agreement

Validation labels tell us who is right on known data. Test ZIPs tell us whether SAM and TinyViT produce meaningfully different submission masks. Strong disagreement without validation complementarity is risk, not signal.

In [ ]:
def read_zip_masks(path):
    if Image is None:
        return None
    path = Path(path)
    if not path.exists():
        print('missing ZIP:', path)
        return None
    masks = {}
    with zipfile.ZipFile(path) as zf:
        for name in sorted(n for n in zf.namelist() if n.endswith('.png')):
            with zf.open(name) as fh:
                masks[Path(name).stem] = np.array(Image.open(fh), dtype=np.uint8)
    return masks

test_masks = {name: read_zip_masks(zip_path(info['run_id'])) for name, info in RUNS.items()}

if test_masks['tiny_vit'] is not None and test_masks['sam_decoder'] is not None:
    assert set(test_masks['tiny_vit']) == set(test_masks['sam_decoder'])
    rows = []
    for name, masks in test_masks.items():
        pixels = np.concatenate([m.ravel() for m in masks.values()])
        counts = np.bincount(pixels, minlength=5)
        row = {'model': name, 'patches': len(masks), 'flat_patches': sum(np.unique(m).size == 1 for m in masks.values())}
        row.update({f'class_{i}_frac': counts[i] / counts.sum() for i in range(5)})
        rows.append(row)
    display(pd.DataFrame(rows))

    ids = sorted(test_masks['tiny_vit'])
    agree = []
    for pid in ids:
        a = test_masks['tiny_vit'][pid]
        b = test_masks['sam_decoder'][pid]
        agree.append({'patch_id': pid, 'agreement': float((a == b).mean()), 'mean_abs_delta': float(np.abs(a.astype(np.int16) - b.astype(np.int16)).mean())})
    test_agree = pd.DataFrame(agree)
    display(test_agree.describe())
    display(test_agree.sort_values('agreement').head(20))


## Decision Template

After running the notebook, fill this in before spending a CodaBench submission:

- SAM standalone: reject unless its public score is measured and beats `50.63`.
- SAM in ensemble: consider only if validation sweep improves over TinyViT alone and the improvement is not just exact-accuracy noise.
- SAM follow-up training: only worth another run if the error analysis shows consistent SAM-only gains in classes TinyViT misses. Otherwise prioritize dense ResNet/FPN and seasonal TinyViT.
- If SAM class 2 remains collapsed, the next SAM variant must change the loss or class weighting; do not only train longer.